# Notebook 3: Feature Engineering Pipeline

## Purpose
This notebook **enriches the risk-merged ACLED dataset with three additional feature
groups** and saves a single comprehensive dataset for the multi-model ablation benchmark.
It is purely a data-building step — no models are trained here.

## Datasets
| Dataset | Path | Description |
|---------|------|-------------|
| Input | `data/processed/model_risk_merged.csv` | Output of `merge.ipynb` — 38 features (baseline + 7 risk) |
| Macro source | `data/raw/combined_indicators.csv` | World Bank annual macro indicators, wide format |
| Holiday source | `data/raw/holidays_raw.csv` | Public holiday calendar (88k rows, 233 countries, 1997–2030) |
| Output | `data/processed/model_data_risk_macro_holidays_engineered.csv` | 51 features total |

The output filename encodes exactly what is inside:
- **risk** — 7 CAST risk indicator lags (from `merge.ipynb`)
- **macro** — 3 World Bank indicators (income_inequality, youth_unemployment, inflation), prior-year
- **holidays** — monthly public holiday count (lagged)
- **engineered** — 9 derived features from cross-variable analysis

## Feature groups added here
| Group | Features | Source / logic |
|-------|----------|----------------|
| Macro (3) | income_inequality_py, youth_unemployment_py, inflation_py | World Bank → melted → prior-year shift; NaN imputed with year-median |
| Holidays (1) | n_holidays_lag1 | `holidays_raw.csv` → monthly count → lag 1 |
| Engineered (9) | lag-2s, organized_violence, is_active, battles_x_remote, 3-month rolling avgs | `utils/data_cleaning.build_enhanced_features()` |

## What this notebook does
1. Loads the risk-merged dataset (38 features)
2. Joins World Bank macro indicators (prior-year, with NaN imputation)
3. Adds public holiday counts (lagged)
4. Calls `build_enhanced_features()` for 9 interaction/lag/rolling features
5. Saves the combined 51-feature dataset for `model_comparison.ipynb`


In [12]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # project root → finds config/ and utils/

import numpy as np
import pandas as pd
import pycountry
from functools import lru_cache

from config import settings
from utils.data_cleaning import build_enhanced_features

os.makedirs("../data/processed", exist_ok=True)
print("Setup complete.")


Setup complete.


---
## Step 1: Load Base Dataset

We start from `model_risk_merged.csv`, which already contains the 31 original
ACLED features plus 7 lagged CAST risk indicators (38 features total).
All subsequent steps add columns on top of this base.

In [13]:
BASE_PATH = "../data/processed/model_risk_merged.csv"
df = pd.read_csv(BASE_PATH)

print(f"Base dataset: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Date range: {df['month_year'].min()} → {df['month_year'].max()}")
print(f"Regions: {df['matched_admin1_id'].nunique():,}")
print(f"\nCurrent columns ({df.shape[1]}):")
print(list(df.columns))

Base dataset: 278,820 rows x 51 cols
Date range: 2018-01-01 → 2025-06-01
Regions: 3,097

Current columns (51):
['matched_admin1_id', 'month_year', 'Battles (t-1)', 'Explosions/Remote violence (t-1)', 'Protests (t-1)', 'Riots (t-1)', 'Strategic developments (t-1)', 'Violence against civilians (t-1)', 'Excessive force against protesters (t-1)', 'Agreement (t-1)', 'Explosions/Remote violence_neighbours (t-1)', 'Strategic developments_neighbours (t-1)', 'Protests_neighbours (t-1)', 'Violence against civilians_neighbours (t-1)', 'Battles_neighbours (t-1)', 'Riots_neighbours (t-1)', 'linear_month_trend', 'year', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12', 'quarter_2', 'quarter_3', 'quarter_4', 'importance_weight', 'Battles', 'Explosions/Remote violence', 'Violence against civilians', 'country_code', 'risk_contested_election', 'risk_crop_failure', 'risk_economic_concern', 'risk_ethnic_tension', 'risk_military_coup

---
## Step 2: Macro Economic Indicators (World Bank)

**Source:** `data/raw/combined_indicators.csv`

Three annual indicators from the World Bank are joined to each region-month:
- **income_inequality** (Gini coefficient, 0–100): measures income distribution
  within a country. Higher values indicate greater inequality, which research
  links to higher conflict risk.
- **youth_unemployment** (%): the share of 15–24 year olds who are unemployed.
  Youth unemployment is a well-documented driver of recruitment into armed groups.
- **inflation** (%): CPI-based annual inflation. Economic shocks caused by high
  inflation erode purchasing power and are associated with civil unrest.

**Data structure:** the CSV is wide-format with one row per country and columns
like `income_inequality_2018`, `youth_unemployment_2023`, `inflation_2015`.
We melt to long format, then pivot back to one row per (iso3, year).

**Leakage prevention:** we use **prior-year** values for all three indicators.
For example, a region in 2020 receives the country's 2019 indicator values.
This mirrors real-world forecasting where current-year macro data lags publication.

**NaN handling:**
- Gini is sparsely measured (household surveys are expensive); we back-fill
  and forward-fill within each country before taking the prior-year value.
- Countries absent from the World Bank data or with all-missing Gini are
  imputed with the **global year-median** — the typical value across all
  countries that did report that year.

In [14]:
CI_PATH = "../data/raw/combined_indicators.csv"
ci_raw  = pd.read_csv(CI_PATH)

print(f"Shape: {ci_raw.shape}")
display(ci_raw[['countryiso3code','country_name',
                'income_inequality_2018','youth_unemployment_2018','inflation_2018']].head(6))

Shape: (244, 44)


,countryiso3code,country_name,income_inequality_2018,youth_unemployment_2018,inflation_2018
0,NaN,High income,NaN,10.330232,1.881108
1,NaN,Low income,NaN,NaN,2.791808
2,NaN,Lower middle income,NaN,28.675658,4.176377
3,NaN,Upper middle income,NaN,NaN,2.830149
4,ABW,Aruba,NaN,NaN,3.626041
5,AFE,Africa Eastern and Southern,NaN,NaN,4.720805


In [15]:
# Filter out aggregate rows (income groups, regional blocs — no real ISO3 code)
ci = ci_raw[ci_raw['countryiso3code'].str.match(r'^[A-Z]{3}$', na=False)].copy()
print(f"Real country rows: {len(ci)} (filtered {len(ci_raw) - len(ci)} aggregate rows)")

# Melt wide → long: "income_inequality_2018" → indicator="income_inequality", year=2018
meta_cols  = ['countryiso3code', 'country_name']
value_cols = [c for c in ci.columns if c not in meta_cols]

ci_long = ci.melt(id_vars=meta_cols, value_vars=value_cols,
                  var_name='col', value_name='value')
ci_long['year']      = ci_long['col'].str.rsplit('_', n=1).str[-1].astype(int)
ci_long['indicator'] = ci_long['col'].str.rsplit('_', n=1).str[0]
ci_long = ci_long.drop(columns='col')

macro = ci_long.pivot_table(
    index=['countryiso3code', 'year'], columns='indicator',
    values='value', aggfunc='mean'
).reset_index()
macro.columns.name = None
macro = macro.rename(columns={'countryiso3code': 'iso3'})

print(f"\nMacro table: {macro.shape}")
print("Coverage before fill:")
for col in ['income_inequality', 'youth_unemployment', 'inflation']:
    n = macro[col].notna().sum()
    print(f"  {col}: {n}/{len(macro)} ({n/len(macro)*100:.1f}%)")

Real country rows: 240 (filtered 4 aggregate rows)

Macro table: (3226, 5)
Coverage before fill:
  income_inequality: 1065/3226 (33.0%)
  youth_unemployment: 1626/3226 (50.4%)
  inflation: 3186/3226 (98.8%)


In [16]:
# Extend to 2025 so prior-year values are available for all model months (2018–2025)
all_years = pd.DataFrame(
    [(iso3, yr) for iso3 in macro['iso3'].unique() for yr in range(2010, 2026)],
    columns=['iso3', 'year']
)
macro_full = all_years.merge(macro, on=['iso3', 'year'], how='left')
macro_full = macro_full.sort_values(['iso3', 'year'])

# Within each country: back-fill Gini (slow-moving survey data), then forward-fill all
for col in ['income_inequality', 'youth_unemployment', 'inflation']:
    macro_full[col] = macro_full.groupby('iso3')[col].transform(
        lambda x: x.bfill().ffill()
    )

# Prior-year shift: the 2020 value for a country becomes the 2021 predictor
for col in ['income_inequality', 'youth_unemployment', 'inflation']:
    macro_full[f'{col}_py'] = macro_full.groupby('iso3')[col].shift(1)

macro_join = macro_full[['iso3', 'year',
                          'income_inequality_py', 'youth_unemployment_py', 'inflation_py']]
print("Macro feature table built.")
display(macro_join[macro_join['iso3'] == 'UKR'].tail(8))

Macro feature table built.


,iso3,year,income_inequality_py,youth_unemployment_py,inflation_py
3608,UKR,2018,26.0,16.47,14.438323
3609,UKR,2019,26.1,16.47,10.951856
3610,UKR,2020,26.6,16.47,7.886717
3611,UKR,2021,25.6,16.47,2.732492
3612,UKR,2022,25.6,16.47,9.363139
3613,UKR,2023,25.6,16.47,20.183637
3614,UKR,2024,25.6,16.47,12.849022
3615,UKR,2025,25.6,16.47,12.849022


In [17]:
# Apply the same ACLED → ISO3 remapping used in risk_merge.py.
# ACLED uses non-standard codes PSX (Palestine) and SDS (South Sudan);
# the World Bank uses PSE and SSD respectively. Without this mapping,
# Gaza and South Sudan regions would have no macro data at all.
_ACLED_TO_ISO3 = {"PSX": "PSE", "SDS": "SSD"}

df['iso3'] = df['matched_admin1_id'].str.split(' - ').str[0].replace(_ACLED_TO_ISO3)
df['year'] = pd.to_datetime(df['month_year']).dt.year

df = df.merge(macro_join, on=['iso3', 'year'], how='left')

MACRO_FEATURES = ['income_inequality_py', 'youth_unemployment_py', 'inflation_py']

# Impute remaining NaN with global year-median.
# NaN after the join comes from two sources:
#   (1) ~33 ISO3 codes absent from World Bank entirely (territories, closed states)
#   (2) ~29 countries present in WB data but with zero Gini observations
# Year-median imputation assigns these countries the "typical" value for that year,
# which is a defensible and widely used approach for country-level macro data.
for col in MACRO_FEATURES:
    year_medians = df.groupby('year')[col].median()
    mask = df[col].isna()
    df.loc[mask, col] = df.loc[mask, 'year'].map(year_medians)
    df[col] = df[col].fillna(df[col].median())   # final fallback

print("Macro indicators joined and imputed:")
for col in MACRO_FEATURES:
    n_nan = df[col].isna().sum()
    print(f"  {col}: {df[col].notna().sum():,}/{len(df):,} — {n_nan} NaN remaining")
print(f"\nDataset shape: {df.shape}")

Macro indicators joined and imputed:
  income_inequality_py: 278,820/278,820 — 0 NaN remaining
  youth_unemployment_py: 278,820/278,820 — 0 NaN remaining
  inflation_py: 278,820/278,820 — 0 NaN remaining

Dataset shape: (278820, 55)


---
## Step 3: Public Holiday Calendar

**Source:** `data/raw/holidays_raw.csv`  (88,957 rows · 233 countries · 1997–2030)

Public holidays mark days when government, courts, and markets are closed.
For conflict forecasting they serve as a calendar control: attacks, protests,
and ceasefire violations often cluster around national holidays or religious
observances. Including holiday density prevents the model from confusing
seasonal political cycles with structural conflict escalation.

**Feature created:**
- `n_holidays_lag1` — number of public holidays in the **prior month**.
  We lag by one month to stay consistent with all other t-1 features and to
  avoid any theoretical leakage (even though holiday calendars are public
  knowledge, lagging maintains a clean design).

**Country name → ISO3 mapping:** the CSV uses country names (e.g. "Syria",
"Congo DR") rather than codes. We use `pycountry` to resolve them, with a
small manual override dict for names that pycountry does not recognise.

In [18]:
HOL_PATH = "../data/raw/holidays_raw.csv"
hol_raw  = pd.read_csv(HOL_PATH)

print(f"Shape: {hol_raw.shape}")
display(hol_raw.head(4))

@lru_cache(maxsize=None)
def name_to_iso3(name):
    """Map a country name string to its ISO 3166-1 alpha-3 code."""
    overrides = {
        'Congo DR': 'COD', 'U.S. Virgin Islands': 'VIR', 'Kosovo': 'XKX',
        "Cote d'Ivoire": 'CIV', 'Iran': 'IRN', 'South Korea': 'KOR',
        'North Korea': 'PRK', 'Russia': 'RUS', 'Syria': 'SYR',
        'Bolivia': 'BOL', 'Venezuela': 'VEN', 'Tanzania': 'TZA',
        'Vietnam': 'VNM', 'Laos': 'LAO', 'Moldova': 'MDA',
        'Palestinian Territory': 'PSE', 'Taiwan': 'TWN', 'Brunei': 'BRN',
    }
    if name in overrides:
        return overrides[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

hol = hol_raw.copy()
hol['iso3']  = hol['Country'].apply(name_to_iso3)
hol['date']  = pd.to_datetime(hol['Date'], errors='coerce')
hol['year']  = hol['date'].dt.year
hol['month'] = hol['date'].dt.month

unmapped = hol[hol['iso3'].isna()]['Country'].unique()
print(f"\nUnmapped countries ({len(unmapped)}): {unmapped[:10]}")

hol_monthly = (
    hol.dropna(subset=['iso3', 'date'])
    .groupby(['iso3', 'year', 'month'])
    .size()
    .reset_index(name='n_holidays')
)
print(f"Monthly holiday counts: {hol_monthly.shape}")
print(f"Mean per country-month: {hol_monthly['n_holidays'].mean():.2f}")

Shape: (88957, 3)


,Country,Date,Holiday
0,Afghanistan,2014-01-14,The Prophet's Birthday
1,Afghanistan,2014-02-15,Liberation Day
2,Afghanistan,2014-04-28,Afghan Victory Day
3,Afghanistan,2014-05-01,Labor Day



Unmapped countries (14): ['Cape Verde' 'Falkland Islands' 'Ivory Coast' 'La Réunion' 'Macau'
 'Micronesia' 'Saint Helena' 'Saint Martin' 'Sint Maarten' 'Swaziland']
Monthly holiday counts: (47519, 4)
Mean per country-month: 1.77


In [19]:
df['month'] = pd.to_datetime(df['month_year']).dt.month

df = df.merge(hol_monthly, on=['iso3', 'year', 'month'], how='left')
df['n_holidays'] = df['n_holidays'].fillna(0).astype(int)

# Lag by 1 month within each region
df = df.sort_values(['matched_admin1_id', 'month_year'])
df['n_holidays_lag1'] = (
    df.groupby('matched_admin1_id')['n_holidays'].shift(1).fillna(0)
)

HOLIDAY_FEATURES = ['n_holidays_lag1']

hol_covered = (df['n_holidays'] > 0).sum()
print(f"Rows with at least 1 holiday in that month: "
      f"{hol_covered:,}/{len(df):,} ({hol_covered/len(df)*100:.1f}%)")
print(f"Average n_holidays per row: {df['n_holidays'].mean():.3f}")
print(f"Dataset shape: {df.shape}")

Rows with at least 1 holiday in that month: 178,678/278,820 (64.1%)
Average n_holidays per row: 1.245
Dataset shape: (278820, 58)


---
## Step 4: Engineered Features

**Source:** `utils/data_cleaning.build_enhanced_features()`

These 9 features were derived from a cross-variable relationship analysis
documented in `notebooks/models.ipynb`. The key findings that motivated them:

| Feature | Motivation |
|---------|-----------|
| Battles (t-2), Remote (t-2), VaC (t-2) | Lag-2 Spearman r ≈ 0.70 (autocorrelation barely decays) |
| organized_violence (t-1) | CAST's own combined target; captures aggregate threat level |
| is_active (t-1) | Binary flag that helps the model split true zero-conflict from low-activity |
| battles_x_remote (t-1) | Co-escalation: remote violence spikes 13× when Battles > 3 |
| Battles_3mo_avg (t-1) | Rolling average r = 0.717 with Battles — stronger than any single lag |
| Remote_3mo_avg (t-1) | Same rationale for remote violence |
| VaC_3mo_avg (t-1) | Same for violence against civilians |

The function is defined in `utils/data_cleaning.py` to keep the logic versioned
and reusable outside notebooks.

In [20]:
# Generate engineered features from model_data.csv (the original source).
# We do NOT use merged_df because build_enhanced_features needs only the
# base event counts — it doesn't interact with risk or macro columns.
base_df = pd.read_csv("../data/processed/model_data.csv")
enhanced_source = build_enhanced_features(base_df)

ENGINEERED_COLS = [
    'Battles (t-2)', 'Explosions/Remote violence (t-2)',
    'Violence against civilians (t-2)',
    'organized_violence (t-1)', 'is_active (t-1)', 'battles_x_remote (t-1)',
    'Battles_3mo_avg (t-1)', 'Remote_3mo_avg (t-1)', 'VaC_3mo_avg (t-1)',
]

available_eng = [c for c in ENGINEERED_COLS if c in enhanced_source.columns]
print(f"Engineered features generated: {len(available_eng)}/{len(ENGINEERED_COLS)}")
for c in available_eng:
    print(f"  {c}")

Engineered features generated: 9/9
  Battles (t-2)
  Explosions/Remote violence (t-2)
  Violence against civilians (t-2)
  organized_violence (t-1)
  is_active (t-1)
  battles_x_remote (t-1)
  Battles_3mo_avg (t-1)
  Remote_3mo_avg (t-1)
  VaC_3mo_avg (t-1)


In [21]:
enh_join = enhanced_source[['matched_admin1_id', 'month_year'] + available_eng].copy()

df['month_year']       = df['month_year'].astype(str)
enh_join['month_year'] = enh_join['month_year'].astype(str)

df = df.merge(enh_join, on=['matched_admin1_id', 'month_year'], how='left')
for col in available_eng:
    df[col] = df[col].fillna(0)

ENGINEERED_FEATURES = available_eng
print(f"Dataset shape after engineered features: {df.shape}")

Dataset shape after engineered features: (278820, 67)


---
## Step 5: Save Combined Dataset

The final dataset contains every feature group stacked together. The filename
encodes the feature groups present so future readers can identify its contents
without having to open it.

In [22]:
# ── Final feature inventory ────────────────────────────────────────────────
from utils.risk_merge import RiskIndicatorMerger
_merger = RiskIndicatorMerger(lag=1)
RISK_FEATURES = _merger.risk_cols_lagged_

all_feature_groups = {
    "Original 31 predictors": settings.predictors,
    "Risk indicators (t-1)":  RISK_FEATURES,
    "Macro indicators (py)":  MACRO_FEATURES,
    "Holiday features":       HOLIDAY_FEATURES,
    "Engineered features":    ENGINEERED_FEATURES,
    "Targets":                settings.targets,
}

print("=" * 60)
print("FINAL DATASET — FEATURE INVENTORY")
print("=" * 60)
total_features = 0
for group, cols in all_feature_groups.items():
    avail = [c for c in cols if c in df.columns]
    total_features += len(avail)
    print(f"  {group:30s}: {len(avail):3d} features")
print(f"  {'─'*40}")
print(f"  {'TOTAL (excl. targets)':30s}: {total_features - len(settings.targets):3d} features")
print(f"  {'Targets':30s}: {len(settings.targets):3d}")
print(f"\n  Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"  Date range:    {df['month_year'].min()} → {df['month_year'].max()}")
print(f"  Regions:       {df['matched_admin1_id'].nunique():,}")

# ── Save ──────────────────────────────────────────────────────────────────
OUT_PATH = "../data/processed/model_data_risk_macro_holidays_engineered.csv"
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")

FINAL DATASET — FEATURE INVENTORY
  Original 31 predictors        :  31 features
  Risk indicators (t-1)         :   0 features
  Macro indicators (py)         :   3 features
  Holiday features              :   1 features
  Engineered features           :   9 features
  Targets                       :   3 features
  ────────────────────────────────────────
  TOTAL (excl. targets)         :  44 features
  Targets                       :   3

  Dataset shape: 278,820 rows x 67 cols
  Date range:    2018-01-01 → 2025-06-01
  Regions:       3,097

Saved: ../data/processed/model_data_risk_macro_holidays_engineered.csv
